# AwareLiquid · Cloud-Inject Uplift on Qwen-2.5-1.5B + Phase 5b adapter

Runs `scripts/bench_cloud_inject_uplift.py --backend hf` on 30 factual questions, with the Phase 5b adapter loaded. Quantifies how much accuracy lift the cloud-inject template gives on a real model.

Phase 5b kernel output is mounted as input dataset `/kaggle/input/awareliquid-phase-5b/`.

Wall time: ~10-20 min on T4.

## 0 · Pin torch for sm_60 + sm_75 (P100/T4 either way)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
    'torch==2.4.1', 'torchvision==0.19.1',
    '--index-url', 'https://download.pytorch.org/whl/cu121'])
print('torch pinned to 2.4.1+cu121')
import sys as _s
if 'torch' in _s.modules:
    import os, signal; os.kill(os.getpid(), signal.SIGTERM)

## 1 · Clone M1 + GPU sanity

In [ ]:
import os, subprocess, torch
REPO = 'https://github.com/AwareLiquid/M1.git'
DIR = '/kaggle/working/M1'
if not os.path.exists(DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO, DIR])
os.chdir(DIR)
subprocess.check_call(['git', 'log', '-1', '--oneline'])
cap = torch.cuda.get_device_capability(0)
print('gpu:', torch.cuda.get_device_name(0), f'sm_{cap[0]}{cap[1]}')
print('cuda_smoke:', (torch.randn(4,4,device='cuda') @ torch.randn(4,4,device='cuda')).sum().item())

## 2 · Install deps

In [ ]:
!pip install -q -r requirements.txt accelerate safetensors peft datasets

## 3 · Locate Phase 5b adapter checkpoint

In [ ]:
import glob, os
candidates = sorted(glob.glob('/kaggle/input/**/llama_mt_adapter_001000.pt', recursive=True))
print('checkpoints found:')
for c in candidates: print(' ', c, os.path.getsize(c)//1024, 'KB')
assert candidates, 'no Phase 5b adapter checkpoint mounted'
ADAPTER = candidates[-1]
print('using:', ADAPTER)

## 4 · Run cloud-inject uplift (no-adapter baseline)

In [ ]:
!PYTHONPATH=/kaggle/working/M1 python scripts/bench_cloud_inject_uplift.py \
    --backend hf \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --max_tokens 60 \
    --out /kaggle/working/cloud_inject_uplift_qwen_baseline.json

## 5 · Run cloud-inject uplift (with Phase 5b adapter)

In [ ]:
import os
os.environ['ADAPTER_PATH'] = ADAPTER
!PYTHONPATH=/kaggle/working/M1 python scripts/bench_cloud_inject_uplift.py \
    --backend hf \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --adapter $ADAPTER_PATH \
    --max_tokens 60 \
    --out /kaggle/working/cloud_inject_uplift_qwen_mtadapter.json

## 6 · Summarize both reports

In [ ]:
import json, shutil
from pathlib import Path
for name in ('cloud_inject_uplift_qwen_baseline.json', 'cloud_inject_uplift_qwen_mtadapter.json'):
    p = Path('/kaggle/working') / name
    if not p.exists():
        print('MISSING:', p); continue
    r = json.loads(p.read_text())
    print(f'\n=== {name} ===')
    print(f"backend={r['backend']}  n={r['n_questions']}  wall={r['wall_s']}s")
    print(f"  no_inject_accuracy : {r['no_inject_accuracy']:.3f}")
    print(f"  inject_accuracy    : {r['inject_accuracy']:.3f}")
    print(f"  uplift_abs         : +{r['uplift_abs']:.3f}")
    print(f"  uplift_rel         : {r['uplift_rel']}")
out = Path('/kaggle/working/cloud_inject_artifacts'); out.mkdir(exist_ok=True)
for n in ('cloud_inject_uplift_qwen_baseline.json', 'cloud_inject_uplift_qwen_mtadapter.json'):
    p = Path('/kaggle/working') / n
    if p.exists(): shutil.copy(p, out / n)
archive = shutil.make_archive('/kaggle/working/cloud_inject_uplift_qwen', 'zip', out)
print('\narchive:', archive)